In [3]:
import pandas as pd
import numpy as np

from pathlib import Path

In [4]:
# chemins vers fichiers Data et Images

BASE_DIR = Path.cwd()

if BASE_DIR.name == "Notebooks":
    BASE_DIR = BASE_DIR.parent

DAT_DIR = BASE_DIR / "Data"
IMG_DIR = BASE_DIR / "Images"
DOC_DIR = BASE_DIR / "Docs"

In [5]:
# import matrice étendue

matrix_all_extended = pd.read_csv(DAT_DIR / "LittRu_matrix_all_extended.csv", sep=',', header=0)

In [6]:
# indique que les thèmes sont toutes les colonnes situées après "CutOff"

theme_cols = matrix_all_extended.columns[
    matrix_all_extended.columns.get_loc("CutOff") + 1:
]

**Matrice contenant : les oeuvres, les auteurs ET les thèmes**

In [7]:
# matrice contenant "Works", "Authors", puis les colonnes de thèmes

matrix_WAT = pd.concat(
    [
        matrix_all_extended[["Works", "Author"]],
        matrix_all_extended[theme_cols]
    ],
    axis=1
)

**Construction de la matrice réduite ne contenant que :**
- les thèmes qui se rencontrent dans au mois 10 oeuvres
- les oeuvres qui contiennent au moins 4 thèmes

In [8]:
# matrice des thèmes seuls
matrix_themes = matrix_WAT[theme_cols]

# 1) thèmes présents dans au moins 10 œuvres
theme_counts = matrix_themes.sum(axis=0)

theme_cols_reduced = theme_counts[
    theme_counts >= 10
].index

# 2) œuvres contenant au moins 4 thèmes
# calcul effectué sur la matrice complète initiale des thèmes
work_counts = matrix_themes.sum(axis=1)

works_reduced = work_counts >= 4

# 3) matrice Works / Author / Themes réduite
matrix_WAT_reduced = matrix_WAT.loc[
    works_reduced,
    ["Works", "Author"] + list(theme_cols_reduced)
].copy()

# remise à zéro de l'index
matrix_WAT_reduced = matrix_WAT_reduced.reset_index(drop=True)

**Profil thématique réduit des auteurs**

In [9]:
matrix_reduced_profile = (
    matrix_WAT_reduced
    .groupby("Author")[theme_cols_reduced]
    .sum()
)

matrix_reduced_profile.index.name = "Author"
matrix_reduced_profile.columns.name = "Themes"

**Export de la matrice des données réduites vers Excel**

In [10]:
with pd.ExcelWriter(
    DAT_DIR / "LittRu_AC_matrix_reduced_profile.xlsx"
) as writer:
    matrix_reduced_profile.to_excel(
        writer,
        sheet_name="Matrix"
    )

print(
    "Fichier Excel enregistré :",
    DAT_DIR / "LittRu_AC_matrix_reduced_profile.xlsx"
)

Fichier Excel enregistré : C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_matrix_reduced_profile.xlsx


**Export de la matrice des données réduites au format .CSV**

In [11]:
matrix_reduced_profile.to_csv(
    DAT_DIR / "LittRu_AC_matrix_reduced_profile.csv",
    sep=",",
    index=True,
    encoding="utf-8-sig"
)

print(
    "Fichier CSV enregistré :",
    DAT_DIR / "LittRu_AC_matrix_reduced_profile.csv"
)

Fichier CSV enregistré : C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_matrix_reduced_profile.csv
